**Review copy (October 2026).** This is David Cyganski's `WignerParticlesSinSpawnV1` (17 April 2020) with the bugs listed in [`sinspawn_v1_review.md`](../supplement/sinspawn_v1_review.md) fixed and the code ported to Python 3. Every changed line is marked `FIX`. The main loop (the cell after *Here is the main propagation and spawing loop*) is the only cell substantially rewritten, and it keeps the original data structures and names. Two cells are added: a fixed random seed in the constants cell, so that a re-run reproduces the numbers quoted in the review, and, at the end, a comparison with the exact (Schrödinger) solution of the same problem.

This copy has its outputs stripped. The executed copy, with all outputs (about 23 MB, mostly the plotly 3-D surfaces), is on the `output` branch: [rendered](https://nbviewer.org/github/billpage/wpmw/blob/output/notebooks/sinspawn_v1_fixed.ipynb) · [file](https://github.com/billpage/wpmw/blob/output/notebooks/sinspawn_v1_fixed.ipynb). GitHub does not display notebooks of that size, so use the first link to read it.

# WignerConvWithParticles/WignerParticlesSinSpawn

A continuation of WignerConvWithParticles/WignerParticlesNoSpawn but introduces spawning of particle pairs due to a quadratic potential function with an additional sinusoidal component. For that reason we will tack on the new version information after the NoSpawn versions below.

### WignerParticlesNoSpawn

WignerConvWithParticles/WignerParticlesNoSpawn - D.Cyganski March 30, 2020. Transition WignerConvV2a into a particle only, no collision numpy array based (for speed) for exploration of solutions involving linear and quadratic fields.

WignerParticlesNoSpawnV1: D.Cyganski April 4, 2020. Gaussian Wavepacket with non-zero initial average momentum propagating with zero potential field environment. Complete.

WignerParticlesNoSpawnV2: D.Cyganski April 6, 2020. Gaussian Wavepacket with zero initial average momentum propagating in a linear potential field (constant force) environment. Completed and fully functional on April 9, 2020.

WignerParticlesNoSpawnV3: D.Cyganski April 9, 2020. Gaussian Wavepacket with zero initial average momentum propagating in a Quadratic potential field (linear force) environment. Completed and fully functional on April 10, 2020.

### WignerParticleSinSpawn 

WignerParticlesSinSpawnV1: D.Cyganski April 17,2020. Gaussian Wavepacket with zero initial average momentum propagating in a Quadratic plus sinusoidal potential environment. This creates an asymmetric quantum well which results in a solution which is one example of an anharmonic oscillator.

V1 specific notes: This version was frozen when the most important functions had been made functional, but prior to completion. The simulation executes and produces apparently correct (no verification against standard solution methods yet) results and graphs the trajectories and histograms of the outcomes. Particle creation, deletion and curve kinks are all treated. Trajectories are handled correctly and plotted as segments over the lifetime of any of the selected tracks. Here is what is not done yet:

  1. No garbage collection and reuse of particle array entries after annihilation
  2. No array growth if the originally reserved particle array space is exceeded.
  3. No comparison to Schrodinger solution of same problem
  4. No physical scaling in the particle creation rate constant
  5. I need to verify the handling of the sinusoidal potential wavelength scaling.


NOTE: Some of the more compute intensive 3-D graphics have been turned off with the first cell code line: %%script false

Simply comment out that line in those cells to reactivate the graphic.


### Earlier Development History

Above code line derived from earlier code line:

Derived from: WignerConvV2a, version history of which was:

WignerConV1 - D.Cyganski May 2019 - 19 June 2019. Frozen at point that captures a well working example of collisionless (Newtonian) paricle propagation, plotting of outcomes, and implementation of Wigner potential computation as preparation for collision processes.  Next version incorporates the collision process.

WignerConvV2 - D.Cyganski 20 June 2019 - Introduces collision process and was the subject of nearly a month of troubleshooting which spawned several Maxima scripts that paralleled the processing with analytic solutions that could be used for comparison based troubleshoots. Eventually several errors were located, some being incorrect implementation of some math, some being accidental use of incorrect indices in complex loops, and one case of falling into an the object-orientation trap of not sufficiently deeply cloning an array. As of completion this version "appears to work." That is, the general behaviour looks good but the outcomes are two low-resolution to really tell since the processing time is long enough that I ran only 1E5 particles. Next step will implement multicore processing so that I can use more particles and better confirm the outcomes.

WignerConvV2a - D. Cyganski 6 Jan 2020 - Same as WignerConvV2 but DT, TimeSteps, NX, NKX, DKX all changed to agree with values used in WignerFluid6CN after lessons learned about needed resolution for numerical convergence. 


In [ ]:
%matplotlib inline
import math
import numpy as np
import multiprocessing 
import plotly.graph_objs as go
import time
import matplotlib.pyplot as pl

In [ ]:
# Prepare plotly graphics environment for notebooks
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=True)

In [ ]:
# specify constants
hbar = 1.05457266E-34   # planck's constant
m = 9.1093897E-31      # particle mass
q_e = 1.60217662E-19;
PI = np.pi
np.random.seed(20200417)  #REVIEW: fixed seed so that a re-run reproduces the review's numbers

#IPNUM = int(1E5) #Initial Number of particles targeted for simulation, actual number 
# will depend on discretization and vary slightly from this and will be called INUM
IPNUM = int(1E3) #Smaller number of IPNUM used for troubleshooting of basic loop operations


dt = 10E-17  #Size of Time step
TimeSteps = 10000 #Usual time window for quadratic potential problem
#TimeSteps = 2060 #Small number of time steps for initial troubleshooting


# specify range in x coordinates
#Length of x domain
LX = 4E-7 #Total width
NX = 2**10 #was 11 #Number of x samples and number of k samples
dx = LX/NX
x = dx * (np.arange(NX) - 0.5 * NX) #centered around zero

#specify k coordinates
NK = NX//2 #FIX: integer for Python 3
##will result in same number of momentum samples as position samples since k is double sided
dk = 2*np.pi/LX #Nyquist determined relationship compatible with FFT relationship
k = (np.array(range(-NK,NK)))*dk #two sided with zero at k[NK+1]
hmt=(hbar/m)*dt #handy constant for later evolution expressions

Above we formed the x and k coordinate values

The discrete Wigner distribution will be maintained in an NX by 2*NK array

We fill the x(i) array with the center values of each box to use as exemplars 
  
``` 
x(0)=-LX/2, x(1)=-LX/2+dx ... x(NX-1) = -LX/2 +(NX-1)dx = LX/2 - dx

note that x(NX/2) = 0 is the center indexed value.
```

We fill the k(i) array similarly, with a total of 2 NK values. For FFT based processing which is introduced
later in this notebook we need to choose NK = NX/2, so that the total number of k values is the same as the
total number of x values, NX.

Since we will be using an even numbered array to hold our k values (and slices of the 2D distribution later) we will have one more sample in one direction than the other about zero. We choose this to be on the negative side or the array, corresponding to the usual choice when representing FFT samples so that the use of FFT processing will require no modifications.  Hence our k array corresponds to samples:
  
```
k(0)=-NK*dk, ..., k(NK/2)=0, ..., k(2*NKX) = (NK-1)*dk 
```

## Form the Initial Wigner Distribution

In [ ]:
# specify initial momentum and quantities derived from it
#k0 = 100.0*dk #Value used in WignerParticlesNoSpawnV1, non-zero k0, zero potential demo
k0 = 0.0*dk #Value used in WignerParticlesNoSpawnV2, zero k0, non-zero potential demo
p0 = k0*hbar
v0 = p0 / m

# mean location of initial Gaussian wave packet
x0 = -0.125*(LX/2.0)
#Define standard deviation of Gaussian packet probability density
a = 2.0E-9 

#Declare some of the strucures we need
Wxk = np.zeros([NX,2*NK]); #Holds the initial Wigner Distribution in normed probability form


In [ ]:
def GaussianEvolution(t, x, x0, a, k0, hbar, m):
    x=x-x0
    return ((2*np.pi)**(-1.0/4.0)/np.sqrt(a+(hbar/(2.0*a*m))*1j*t))*np.exp(1j*(k0*x-hbar*k0*t/(2*m)))*np.exp(-(x-hbar*k0*t/m)**2/(4.0*(a**2+1j*hbar*t/(2.0*m))))

In [ ]:
# Note: values of hbar and m given below are immaterial since t=0 for initial packet formation
def gauss_x(x, a, x0, k0):
    """
    a gaussian wave packet of stddev = a, centered at x0, with momentum k0
    """ 
    return GaussianEvolution(0, x, x0, a, k0, 1, 1)

Recall that for a Gaussian pulse with minimum uncertainty $\sigma_0=a$ the momentum space pulse will have $\sigma_p=\hbar/(2 a)$; Furthermore, for expression in terms of k rather than p, we must convert the momentum standard deviation to wavenumber units also, hence $\sigma_k = \sigma_p/\hbar = 1/(2 a)$

The definition of the Wigner Transform of the Schrodinger wave function is given by

$$W(x,k) = \frac{1}{\pi} \int_{-\infty}^{\infty} \psi^*(x+s)\,\,\psi(x+s)\,\, e^{2 i k s} ds$$

In my Maxima worksheet GaussianWignerFunc.wxmx I symbolically find the Wigner distribution for a propagating Gaussian packet (defined in wavefunction expression from Eq. 4.7.2 of Holland's Quantum Theory of Motion.
Let a be the standard deviation defining the width of the Gaussian in the x domain at t=0.
k0, the mean wavenumber, where k is defined as $k=p/\hbar$, and gives packet center of mass a velocity of $u=\hbar k_0/m$

$$ \frac{1}{\pi} {e^ {- {{\left(4\,k_{0}^2-8\,k\,k_{0}+4\,k^2\right)\,m^2\,{a}
 ^4+m^2\,x^2+2\,{\hbar}\,k\,m\,t\,x+{\hbar}^2\,k^2\,t^2}\over{2
 \,m^2\,{a}^2}} }}$$

In [ ]:
def GaussianWignerPulse(x,k,a,x0,k0,m,hbar,t):
    return np.exp(-(((4*k0**2-8*k*k0+4*k**2)*m**2*a**4+m**2*(x-x0)**2+2*hbar*k*m*t*(x-x0) +hbar**2*k**2*t**2)/(2.0*m**2*a**2)))/(np.pi)

In [ ]:
#Compute the initial Wigner Function and verify that it is normalized
 
for i in range(0,NX):
  for j in range(2*NK):
    Wxk[i,j] = GaussianWignerPulse(x[i],k[j],a,x0,k0,m,hbar,0)
np.sum(Wxk)*dx*dk

Use plotly to plot the distribution fuction. 

Note: plotly associates the x and y coordinates with the column and row indices of the z matrix, respectively - the opposite of our association above! So, don't forget to transpose z matrix. This warning is especially important because plotly will not throw an error for a mismatch of the sixes of the x and y coordinate label arrays and the sizes of the row and col dimensions of z - it will generate perfectly nice looking arrays in either case. This fact lost me a day to troubleshooting! 

In [ ]:
%%script false --no-raise-error
surf = go.Surface(x=x,y=k,z=np.transpose(Wxk), colorbar = dict(exponentformat = 'power')) 
Wdata = [surf] 
Wlayout = go.Layout(
                    scene = dict(
                    xaxis = dict(
                        title='x (m)', exponentformat = 'power'),
                    yaxis = dict(
                        title='k (momentum)', exponentformat = 'power'),
                    zaxis = dict(
                        title='Initial Wigner Distribution', exponentformat = 'power'),),
                    width=700,
                    margin=dict(
                    r=20, b=10,
                    l=10, t=10)
                    
)
    
Wfig = go.Figure(data=Wdata, layout=Wlayout)
iplot(Wfig)

In [ ]:
#normalization of the initial conditions (set double integral with cell size weighting of Wxk to 1)
norm = np.sum(Wxk)
Wxk/=(norm*dx*dk)
np.sum(Wxk)*dx*dk

Verify that our expression reproduced the specified x marginal statistics

In [ ]:
gmean = np.sum(Wxk*x[:, None])*dx*dk
gvar = np.sum(Wxk*((x*x)[:,None]))*dx*dk - gmean**2
gstd = np.sqrt(gvar)
print("Mean = %e, Var = %e, StdDev = %e" % (gmean, gvar,gstd))

Also verify that our expression reproduced the specified k marginal statistics.

Recall that for a Gaussian pulse with minimum uncertainty $\sigma_0=a$ the momentum space pulse will have $\sigma_p = \hbar \sigma/2$ or expressed in k, $\sigma_k=\sigma_0/2$. We also verify this below.

In [ ]:
kmean = np.sum(k*Wxk)*dx*dk
kvar = np.sum((k)*(k)*Wxk)*dx*dk - kmean**2
kstd = np.sqrt(kvar)
print("Mean = %e, Var = %e, StdDev = %e, analytic StdDev = 1/2a =  %e" % (kmean, kvar,kstd,1.0/(2.0*a)))

In [ ]:
#Calculate the ParticleDeltaW variable to determine absolute mass per particle in order to find particles 
#per cell

#Begin by finding the absolute total probability mass of pos and neg particles
#because if W has negative entries, some of our particles need to go there and hence sign does not matter 
#when counting out the distribution.
abs_dist_total = np.sum(np.abs(Wxk))

#Find amount of density associated with a single particle to get
# approximately (due to round off effects) a desired IPNUM (initial particle number)
# particles in total assigned to the distribution function matrix
ParticleDeltaW=abs_dist_total/IPNUM
print("Particle delta W = %f" % (ParticleDeltaW))

## Explore the data structures and search methods for particle arrays

We will be storing x,k, xbox, kbox and a flag value indicating that this particle slot is currently being used.

In [ ]:
ptest=np.array([[2.2,3.1,2,3,1],[3.2,1.1,3,1,1],[6.6,3.6,7,4,1],[6.9,3.8,7,4,1],[7.2,8.1,7,8,1]])
display(ptest)

In the section below where we have updated all particle x and k values during the pure propagation step, we will need to rapidly search for particles in specific boxes or in boxes that have changed from just before propagaion, returning the particle's index and numbers of particles that match. Let's build a few concepts for such searches here that can be applied later.

First, examples that show finding particles that have two properties (x and k) that match two given values:

In [ ]:
matches=np.where((ptest[:,2] == 5) & (ptest[:,3]==4))[0]
print(matches)
print("matches.size = %d" %(matches.size))
matches=np.where((ptest[:,2] == 7) & (ptest[:,3]==4))[0]
print(matches)
print("matches.size = %d" %(matches.size))
matches=np.where((ptest[:,2] == 7) & (ptest[:,3]==8))[0]
print(matches)
print("matches.size = %d" %(matches.size))

We can construct tests for particular values in a column as shown above, or, either or simultaneous matches for more than one column. Note: Boolean operations on lists and arrays in Python are NOT WHAT THEY SEEM TO BE!  See my notes on the subject. Below, the correct operator when comparing numpy arrays and combining logical array outcomes are the bitwise operators & and | and not "and" and "or". 

Now demonstrate how we can find changes between two such arrays of particles in xbox and kbox values and obtain a list of the particle index for those particles having those changes.

In [ ]:
ptest_changed=np.array([[2.2,3.1,2,3,1],[3.2,1.1,4,1,1],[6.6,3.6,7,5,1],[6.9,3.8,8,3,1],[7.2,8.1,7,8,1]])
display(ptest_changed)

In [ ]:
matches=np.where(ptest[:,2] != ptest_changed[:,2])[0]
display(matches)
matches=np.where(ptest[:,3] != ptest_changed[:,3])[0]
display(matches)
matches=np.where((ptest[:,2] != ptest_changed[:,2]) | (ptest[:,3] != ptest_changed[:,3]))[0]
display(matches)

In [ ]:
#Create a helper function for matching the boxed x and boxed k values of one row to an array, live values only
def fboxmatch(row,particles):
    matches=np.where((particles[:,2] == row[2]) & (particles[:,3]==row[3]) & (particles[:,4]==1))[0]
    return matches
    
fboxmatch([7.3, 8.4, 7. , 8. , 1. ],ptest)    

In [ ]:
#For readability, we define suggestively named index values for the particle data elements
xIndex = 0
kIndex = 1
xboxIndex = 2 #discrete x location in distribution array
kboxIndex = 3 #discrete k location in distribution array
aliveIndex = 4  #set to 1 if this particle entry is alive, or 0 if empty and reusable entry

We can use indexing over lists to extract columns, for example here are the two boxed value columns.

In [ ]:
display((np.round(ptest[:,(xboxIndex,kboxIndex)])).astype(int))

And we can combine this with multiple assignment to extract all the values and set variables at once:

In [ ]:
xtest, ktest, xboxtest, kboxtest = ptest[1,(xIndex, kIndex, xboxIndex,kboxIndex)]
print(xtest, ktest, xboxtest, kboxtest)

Another need that will arise below is finding the index of a list entry for the entry that matches a given value.

In [ ]:
listofvalues = [1,4,3,6,8,5]
listofvalues.index(6)

## Calculate initial particle number distribution function and initialize array of particles

Demonstration of the structure I will use to store and propagate particles in phase space. 

Each row is a particle and a particle has three attributes:
x location, k momentum, discrete x, discrete k and flag indicating if this array item is alive or not

In demo below, A is array of 2 particles, with row index choosing the particle.

For a zero potential field we can do our propagation step as a numpy vectorized operation in which 
only location is only changing attribute. DemoHMT is our surrogate for the propagation factor $(\hbar/m) dt$

In [ ]:
Atest = np.array([[2.2,3.1,2,3,1],[5.2,7.1,3,1,1],[8.8,9.6,9,10,0]], dtype=np.float32)
DemoHMT = 0.1
#We can now update 
Atest[:,xIndex] = Atest[:,xIndex]+DemoHMT*Atest[:,kIndex]
#The following update of the "box" values is a concept demo here and not the actual update needed below
Atest[:,(xboxIndex,kboxIndex)] = np.round(Atest[:,(xIndex,kIndex)])
#Note that the boxed variables are floats and not int because numpy arrays have uniform element types
display(Atest)

In [ ]:
#This shows how we can divide each column of the extracted submatrix by a different quantization value
vtest=np.array([.5,.25])
Btest=np.round(np.array(Atest[:,(xIndex,kIndex)]/vtest))
display(Btest)
Atest[:,(xboxIndex,kboxIndex)] = np.round(np.array(Atest[:,(xIndex,kIndex)]/vtest))
display(Atest)

Let's use this opportunity to make sure that testing the alive flag will work with it being saved as a numpy array float when comparing to an integer test value.

In [ ]:
display(Atest[:,4] == 1)
display(Atest[:,4] == 0)

We will create numpy object arrays to hold particle index lists below. Here is how we allocated and manipulate them.

In [ ]:
A_2D_array_of_lists=np.empty((3,4),dtype=object)
display(A_2D_array_of_lists)
A_2D_array_of_lists[0,0]=[1,4,8,12]
display(A_2D_array_of_lists)
A_2D_array_of_lists[0,0][2]
A_2D_array_of_lists[0,0].append(9)
display(A_2D_array_of_lists)

Allocate numpy array of zero arrays. Why am I using a numpy array here and not simply a 2d list? Because I will be able to express the advective updates with numpy vectorized operations.  It's also nice that I also prefer numpy indexing notation \[i,j\] to list notation \[i\]\[j\].

In [ ]:
#WNxk holds the initial discretized particle density for purposes of testing/plotting
WNxk = np.zeros( (NX,2*NK), dtype=int)

#WPxkCounts will hold the current count based absolute particle number histogram connected to the list representation
WPxkCounts = np.zeros( (NX,2*NK), dtype=int)

#WPxkSignedMassHist will hold the current count based signed particle histogram connected to the list representation
WPxkSignedMassHist = np.zeros( (NX,2*NK), dtype=int)

#WPxkpLists holds list for each x,k box, of the particle index numbers in the WPxkp array that fall in this box
WPxkpLists = np.empty((NX,2*NK),dtype=object)
#WPxknLists holds list for each x,k box, of the particle index numbers in the WPxkn array that fall in this box 
WPxknLists = np.empty((NX,2*NK),dtype=object)

In [ ]:
IPMULT = 3 #Multiplier for how many more particles than initial particles to allow due to spawing 
#WPxkp holds the array of positive particles, Create an zero array here for assignment below
WPxkp  = np.zeros((int(IPMULT*IPNUM),5),dtype=float)
#WPxkn holds the array of negative particles, Create an zero array here for assignment below
WPxkn  = np.zeros((int(IPMULT*IPNUM),5),dtype=float)

In [ ]:
ppnum=0 #index of current positive particle array element to be created in the current step
npnum=0 #index of current negative particle array element to be created in the current step
for i in range(0,NX):
      for j in range(0,2*NK):
        CellParticleNum=int(math.floor((abs(Wxk[i,j])/ParticleDeltaW)+0.5))
        #creates the new local particles in the (i,k)-th phase-space cell
        #the particles are uniformly distributed in space
        #WNxk holds the discretized particle density for purposes of testing/plotting
        WNxk[i,j]=np.sign(Wxk[i,j])*CellParticleNum
        #but what we really need for this implementation of the convolutional signed particle
        #Wigner algorithm is the array of particle attribute arrays (one per particle), WPxkp and WPxkn,
        #and an array indexed by i and j phase space indices that points back to the particle indices in
        #the particle attribute arrays. This latter employs a 2d numpy array of index lists.
        
        #We begin initialization of the particle index lists here by creating the empty index lists
        WPxkpLists[i,j] = []
        WPxknLists[i,j] = []
        
        #Create CellParticleNum entries and insert then into the array for this phase space cell
        for cpnum in range(0,CellParticleNum):
            #Assign an initial random x position and k momentum to each particle 
            #equiprobabilistically distributed within the the phase space cell in which it falls
            #using MKS units obtained by using our x and k arrays defined earlier
            rxk = np.random.rand(2)-0.5
            xrand= x[i]+rxk[0]*dx 
            krand= k[j]+rxk[1]*dk
            #Create the particle objects. Each particle has attributes [x,k,m]
            #where m is the mass sign 
            if np.sign(Wxk[i,j]) >= 0.0:
                WPxkp[ppnum,:] = [xrand, krand, np.round(xrand/dx)+NX/2,np.round(krand/dk)+NK, 1]
                #The following line is just a test to make sure that I am recovering xk box indices correctly
                if (i != int(WPxkp[ppnum,xboxIndex])) or (j != int(WPxkp[ppnum,kboxIndex])): print(i,j,WPxkp[ppnum,xboxIndex],WPxkp[ppnum,kboxIndex])
                WPxkpLists[i,j].append(ppnum) #add index of above particle to xk indexed array of index lists
                if ppnum < IPNUM:
                    ppnum+=1  #set ppnum for next assignment
            else: 
                WPxkn[npnum,:] = [xrand, krand, np.round(xrand/dx)+NX/2,np.round(krand/dk)+NK, 1]
                WPxknLists[i,j].append(npnum) #FIX (bug G): was WPxkpLists #add index of above particle to xk indexed array of index lists
                if npnum < IPNUM:
                    npnum+=1  #set ppnum for next assignment
#the number of particles will be very close to but not necessarily equal to IPNUM, so we potentially truncated
#the last few particles in previous line.

#Previously we redefined the length of these arrays possibly yank the last few empty array entries in WPxkm
#WPxkp=WPxkp[0:ppnum,:]
#WPxkn=WPxkn[0:npnum,:]

#To accomodate growth up to some multiple of the initial IPNUM particles due to spawning, we will not 
#truncate these arrays any longer, but rather defer to whatever the current values of ppnum and npnum are.

In [ ]:
print("ppnum = %d, npnum = %d" %(ppnum, npnum))

In [ ]:
%%script false --no-raise-error
#For an adundance of caution, let's check the discretized histogram WNxk
surf = go.Surface(x=x,y=k,z=np.transpose(WNxk), colorbar = dict(exponentformat = 'power')) 
Wdata = [surf] 
Wlayout = go.Layout(
                    scene = dict(
                    xaxis = dict(
                        title='x (m)', exponentformat = 'power'),
                    yaxis = dict(
                        title='k (momentum)', exponentformat = 'power'),
                    zaxis = dict(
                        title='Initial Wigner Distribution', exponentformat = 'power'),),
                    width=700,
                    margin=dict(
                    r=20, b=10,
                    l=10, t=10)
                    
)
    
Wfig = go.Figure(data=Wdata, layout=Wlayout)
iplot(Wfig)

In [ ]:
def MakeWignerHist(PosWParray,NegWParray):
    WHist = np.zeros([NX,2*NK]); #Holds the Wigner Distribution being formed
    for pindex in range(0,ppnum):
        #Recall: x = dx * (np.arange(NX) - 0.5 * NX) #centered around zero 
        #and k = (np.array(range(-NK,NK)))*dk #two sided with zero at k[NK+1]
        iindex = int(np.round(PosWParray[pindex,xIndex]/dx)+NX//2) #FIX (bug I): same box rule as the particles
        jindex = int(np.round(PosWParray[pindex,kIndex]/dk)+NK)
        #print('iindex = %d, jindex = %d' % (iindex, jindex))
        if iindex >= 0 and iindex < NX and jindex >= 0 and jindex < 2*NK:
            WHist[iindex,jindex]+=PosWParray[pindex,aliveIndex]
            #print('WHist(%d,%d) = %d' % (iindex, jindex, WHist[iindex,jindex]))
    for nindex in range(0,npnum):
        #Recall: x = dx * (np.arange(NX) - 0.5 * NX) #centered around zero 
        #and k = (np.array(range(-NK,NK)))*dk #two sided with zero at k[NK+1]
        iindex = int(np.round(NegWParray[nindex,xIndex]/dx)+NX//2)
        jindex = int(np.round(NegWParray[nindex,kIndex]/dk)+NK)
        #print('iindex = %d, jindex = %d' % (iindex, jindex))
        if iindex >= 0 and iindex < NX and jindex >= 0 and jindex < 2*NK:
            WHist[iindex,jindex]-=NegWParray[nindex,aliveIndex]
            #print('WHist(%d,%d) = %d' % (iindex, jindex, WHist[iindex,jindex]))
    return WHist

WPxkSignedMassHist= MakeWignerHist(WPxkp,WPxkn) 

In [ ]:
%%script false --no-raise-error
#This function has not yet been adapted for Spawn code versions

#When working with Wigner distributions that can take on negative values, it is sometimes
#usefull to know the total number of particles disregarding their signs. For those cases we define
#this function. 
def MakeAbsWignerHist(WParray):
    AbsWHist = np.zeros([NX,2*NK]); #Holds the Wigner Distribution being formed
    for pindex in range(0,ppnum):
        #Recall: x = dx * (np.arange(NX) - 0.5 * NX) #centered around zero 
        #and k = (np.array(range(-NK,NK)))*dk #two sided with zero at k[NK+1]
        iindex = int(np.floor(WParray[pindex,xIndex]/dx+NX/2) )
        jindex = int(np.floor(WParray[pindex,kIndex]/dk)+NK )
        #print('iindex = %d, jindex = %d' % (iindex, jindex))
        if iindex >= 0 and iindex < NX and jindex >= 0 and jindex < 2*NK:
            AbsWHist[iindex,jindex]+=1
            #print('WHist(%d,%d) = %d' % (iindex, jindex, WHist[iindex,jindex]))
    return AbsWHist

#WPxkCounts will hold the count based distribution connected to the list representation 
# - used for testing/plotting and during the collision event generation process inside the simulation looop
WPxkCounts= MakeAbsWignerHist(WPxkm) 


In [ ]:
xdat = WPxkp[0:ppnum,xIndex]/dx
ydat = WPxkp[0:ppnum,kIndex]/dk
data = np.column_stack((xdat, ydat))

fig, (ax1, ax2) = pl.subplots(nrows=1, ncols=2,figsize=(8, 4))

ax1.scatter(x=xdat, y=ydat, marker='o', c='r', edgecolor='b')
ax1.set_title('$k$ vs $x$ in units of $dk$ and $dx$')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$k$')

_, bins, _ = ax2.hist(xdat, bins=50, range=[data.min(), data.max()], density=True)
ax2.hist(ydat, bins=bins, alpha=0.5, density=True)
ax2.legend(loc=(0.65, 0.8))
ax2.set_title('Frequency of $x$, $k$ in units of $dk$, $dx$')
ax2.yaxis.tick_right()


In [ ]:
%%script false --no-raise-error
surfWPxkCounts = go.Surface(x=x,y=k,z=np.transpose(WPxkSignedMassHist), colorbar = dict(exponentformat = 'power')) 
dataWPxkCounts = [surfWPxkCounts] 
layoutWPxkCounts = go.Layout(
                    scene = dict(
                    xaxis = dict(
                        title='x (m)', exponentformat = 'power'),
                    yaxis = dict(
                        title='k (momentum)', exponentformat = 'power'),
                    zaxis = dict(
                        title='Wigner Distribution in Particle Counts', exponentformat = 'power'),),
                    width=700,
                    margin=dict(
                    r=20, b=10,
                    l=10, t=10)                    
)
    
figWPxkCounts = go.Figure(data=dataWPxkCounts, layout=layoutWPxkCounts)
iplot(figWPxkCounts)

In [ ]:
#Make copies of pre simulation histograms for later comparison
WPxkCountsOrig = np.copy(WPxkCounts)

#Make a copy of the signed histogram for comparison to outcomes of evolution
WPxkSignedMassHistOrig = np.copy(WPxkSignedMassHist)

rejectcount = 0 #Will count for us how many particles fell outside our finite x,k space
pnum = WPxkSignedMassHistOrig.sum()  #FIX (bug L): pnum was used below but never defined


## Propagation Event Simulation Loop

The following code implements the Newtonian propagation of particles based upon the phase-space coordinates of each particle and the potential field in which it sits, but no collisional spawing. That is, it assumes that the potential field is at most quadratic as any higher order terms in the potential function series expansion would result in particle/anti-particle pair generation (or elevation from out of the vaccuum virtual particle background sea.)

If we were to solve for the general potential field case, the the following would implement the iterations that begin with free space propagation of particles for one time step followed by a convolutional collision process in which there is some probability of a particle spawning a particle, anti-particle pair according to the probability dictated by the Wigner Potentional parameterized convolutional operator.

The Liouville-von Neumann equation that describes the particle propagation process for the Wigner equation can be written in the following form, which expresses the incremental increase in W(x,k) in terms of the advective and collision terms on the right.
$$ \frac{\partial W(x,k)}{\partial t} = - \frac{\hbar k}{m} \frac{\partial W(x,k)}{\partial x} + V_W(x,k) \star_k W(x,k) $$
where $\star_k$ is the binary convolution operator over the variable k, 
$$ V_W(x,k) \star_k W(x,k) = \int_\infty^\infty V_W(x,k-k') W(x,k') dk' $$
where V_W(x,k) is the Wigner potential as defined above as
$$V_W(x,k) = \frac{2Q}{\pi \hbar}\int_0^\infty \sin(2 q k)[\Phi(x+q)-\Phi(x-q)] dq$$
where $\Phi$ is the electric potential and Q is the absolute electron charge in C.
Note that this convolution operation spreads the distribution of particle momenta while leaving them at the same x location. A study of the symmetry properties of the Wigner potential reveals that this spreading can be interpreted as a spawning operation in which each particle remains on its orignal path while spawning a particle-antiparticle pair at equal and opposite momentum deviations from the originating particle's momentum.

If the collisional term is zero, then the first two terms of the evolution equation form a so called advective linear first order partial differential equation that describes how W changes due to the free particle propagation phase alone as was demonstrated in WignerParticlesNoSpawnV1. 

Now the convolutional integral can be expanded in a Taylor series (see my derviation in the note titled: Wigner Particle Model with Curved Trajectories.odt)

$$ V_W(x,k) \star_k W(x,k) = \frac{1}{\hbar} \frac{\partial V}{\partial x}\frac{\partial W}{\partial k} + \sum_{\alpha}^{\infty}
\frac{(-1)^{\alpha}}{\hbar 4^\alpha (2\alpha+1)!}
\frac{\partial^{2\alpha+1} V}{\partial x^{2\alpha+1}} \frac{\partial^{2\alpha+1} W}{\partial k^{2\alpha+1}} $$

When the potential function V(x) is of second order degree in x, or less, we can take advantage of the fact that the series expansion of the convolutional integral is truncated, resulting in a simplified Liouville equation
$$\frac{\partial W(x,k)}{\partial t} = - \frac{\hbar k}{m} \frac{\partial W(x,k)}{\partial x} + \frac{1}{\hbar} \frac{\partial V}{\partial x}\frac{\partial W}{\partial k} $$
leaving us with a differential equation that describes the phase space evolution of a set of particles that follow a curved path motion in which the particles accelerate according to a first order function: that is if $V(x) = V_0+V_1 x + V_2 x^2$ then $\dot{p} = m \ddot{x} = -( V_1 + 2 V_2 x)$ or $$\dot{k} = -\frac{V_1 + 2 V_2 x}{\hbar}$$

In my document "Wigner Particle Model with Curved Trajectories.odt" I find that the inclusion of sinusoidal potential field of the form 
$$V_p \sin(\frac{2 n \pi x}{L}) $$
produces a spawing event of the form
$$V_p \pi \cos\left( \frac{2 \pi n x}{L} \right) \left[ W(x,k+\frac{n \pi}{L})- W(x,k-\frac{n \pi}{L}) \right]$$

Because I wrote the base code for this workbook when tackling the Wigner Liouville distribution evolution problem in which I employed an FFT based Wigner potential computation, the code is hard wired for a value of $dk = \frac{2 \pi}{L}$. Hence the lowest n I can accomodate in the sinusoidal potential case is n = 2.  This is the case we will study below.

## Propagation Processing

Now we perform the Newtonian propagation in phase space, as dictated by the Wigner Particle interpretation of the Wigner/Liouville evolution equation, according to the present location and momentum of each particle in the array. As shown near the top of this worksheet, this is very amenable to processing by a fully vectorized numpy expression. The following code implements the Newtonian propagation of particles but no collisional spawing. That is, it assumes that the potential field is at most quadratic as any higher order terms in the potential function series expansion would result in particle/anti-particle pair generation (or elevation from out of the vacuum virtual particle background sea.)

We will keep an array that captures the track of some of the particles. The constant skipnum defined in the first line below determines what faction of the particles is tracked.

In the case of a quadratic external field, the positions and momentum will change at each step as the momentum of each particle undergoes constant acceleration.

Note: I only implemented the iterate for the case of V1 = 0. I should come back here at some time and bring the linear term back into the potential.  In the case below, the energy in the harmonic oscillator is coming from the fact that the wavepacket has an initial position that is displaced from the center of the potential.

**We will test the quadratic + sin potential case in this, V1, of the WignerParticlesSinSpawn notebook series.**  The potential coefficients are set below.

We will use FracSpawn as the parameter that sets the amount of spawing that ordinarily the amplitude of the sin was determine. However, I don't wish to get bogged down with the derivation of that at the moment and so am taking this indirect approach to setting its value.

In [ ]:
V1=0.0
V2=250.0E6/(NX*dx)
FracSpawn = 0.0001  # per-step spawn probability amplitude; physically FracSpawn = Vp*dt/hbar, i.e. Vp = 0.66 meV here (review)
nharm = 2.0

So far I have modified the old update loop below from the no spawn case to use and update the new postive and negative particle arrays, and to generate a track set from just the positive ones. 

What remains to be done is:
(1) To add spawing,requires extra space to be allowed for in the WPxkp and WPxkn arrays for the addtions (start with 3x the original number of particles), 
(2) Add tracking of newer entries, and combining tracks from both.
(3) Add the spawning
(4) Add cancellation and garbage collection of array entries.

I will use a list of lists to capture the information that I need to plot selected tracks. I need to use this because some tracks will end with a positive-negative cancellation event, others will kink with a positive-negative-pair-creation-replacement event, pure spawning pair-creation (i.e. with no cancellation) event, and in those cases where new tracks are added there will be track array index re-use.  

So will have to keep data on each track array indexed location of the start and end of perhaps many non-contiguous track segments.

In [ ]:
#Demo use of listoflists type of structure that we will use to note track start and end times
listoflists = []
for i in range(0,2):
    sublist = []
    for j in range(0,10):
        sublist.append((i,j))
    listoflists.append(sublist)
print(listoflists)
print(listoflists[0][2])
listoflists[0].append([0,10])
print("Demo use of for loops over list contents") 
for lists in listoflists:
    print(lists)

**Structures for particle track plotting**

Prepare the numpy data structures that will contain the information needed to plot tracks of selected particles, both those initially created, and then later added samples taken of the particle pairs that are generated by interaction with sinusoidal components of the potential field. 

Because these tracks are not continuous, but rather turn on and off, and some come into existence when new particles are added to our system well after initialization, we need several structures to capture track information and to plot the segments later. For positive particles these data structures are:

**pWPxTrack[time index, slot index]:** 
This numpy array structure captures the x position information of selected particles at
every instant of time. We number the selected particles, and hence the collection of tracks for each of those particles by the slot index.  However, since tracks start and end at various times for any given slot index,
we need another structure to define, for efficient plotting treatment, the start and ends of each plottable
time segment within each slot. Hence the structure ....

**ptracklist:**
This python list structure contains two element lists [start, stop] in which start and stop are the time
indices for the stop and stop ends of plottable segments of data in pWPxTrack, above.

**Similarly for the negative particles**

We initially set a single track start/stop entry for each living particle slot spanning the entire simulation run time.  If later a particle in a track slot is annihilated, we revise the end time to the time of this event.
If later a particle is created, we add a new start/stop sublist to the tracklist.

In [ ]:
initiallytrackedparticles = 100
skipnum = int(ppnum/initiallytrackedparticles)
#Prepare to have the number of tracks grow with particle spawning by allowing IPMULT times more tracks
pWPxTrack = np.zeros((TimeSteps,int(IPMULT*IPNUM/skipnum)+1))  #FIX (bug K): room for IPMULT*IPNUM particles
nWPxTrack = np.zeros((TimeSteps,int(IPMULT*IPNUM/skipnum)+1))

#Save the time=0, first, xlocation of the selected tracks that we will be updating over time steps
pWPxTrack[0,0:WPxkp[0:ppnum:skipnum,xIndex].size] = WPxkp[0:ppnum:skipnum,xIndex]
nWPxTrack[0,0:WPxkn[0:npnum:skipnum,xIndex].size] = WPxkn[0:npnum:skipnum,xIndex]

#Build the lists that record the start/end times of the initial tracks we will follow
ptracklist = []
ntracklist = []
for i in range(0,ppnum,skipnum):
    sublist = [[0,TimeSteps]] #All the tracks to be added below start at time=0, end time is the longest possible
    ptracklist.append(sublist)
for i in range(0,npnum,skipnum):
    sublist = [[0,TimeSteps]] #All the tracks to be added below start at time=0, end time is the longest possible
    ntracklist.append(sublist)

In [ ]:
print(pWPxTrack.shape)
print(WPxkp[0:ppnum:skipnum,xIndex].shape)
print(len(ptracklist))

Here is the main propagation and spawing loop that evolves the orginal particles into the new set in accordance with the Stochastic Boltzman particle collision interpretation of the Wigner/Liouville equation. We treat in this version of this code the case of a quadratic+sinusoid potential where the sinusoid is of the form:
$$\sin(2\, \rm{nharm}\, \pi x)$$
where for this run we define the nth harmonic Fourier sin component will be defined by:

In [ ]:
#REVIEW FIXES (2026-10, see WignerParticlesSinSpawnV1_review.md). Changed lines are marked "FIX".
#Define a useful constant here that appears in the quadratic solution case
sqrtvmq = np.sqrt(2.0*V2*m*q_e)
divider_vec = np.array([dx,dk])
offset_vec = np.array([NX/2,NK])
start_time = time.time()
#Keep track of whether we ran out of room in our particle arrays with the current guard multipliers
ppnumexcess = 0
npnumexcess = 0
kinkcount = 0          #FIX (diagnostic): births cancelled at once against an existing particle
annihilcount = 0       #FIX (diagnostic): cancellations on a box change during propagation
kbump = int(round(nharm/2)) #new particle has k bumped plus and minus kbump from current box index


def track_slot(index):
    #FIX (bug H): a particle is tracked only if index % skipnum == 0, and its slot is index // skipnum
    return index // skipnum if index % skipnum == 0 else None


def spawn_pair(parent_sign, xnow, know, xboxnow, kboxnow, gsign, timecounter):
    """FIX (bugs A, C): spawn one pair for a parent of sign parent_sign (+1/-1) where the
    sinusoid's spawn amplitude has sign gsign = sgn(cos(2 nharm pi x/LX)).
    Wigner term:  Gamma(x) [W(k + nharm pi/LX) - W(k - nharm pi/LX)]  with Gamma ~ cos, so the
    child of sign  parent_sign*gsign  goes to k - kbump*dk  and the opposite sign to k + kbump*dk."""
    global ppnum, npnum, ppnumexcess, npnumexcess, kinkcount
    klookdown = max(kboxnow-kbump, 0)            #keep index in array bounds (as in the original)
    klookup = min(kboxnow+kbump, 2*NK-1)
    if parent_sign*gsign > 0:
        kpos, kneg = klookdown, klookup           #positive child below, negative child above
    else:
        kpos, kneg = klookup, klookdown
    dpos = (kpos-kboxnow)*dk                      #momentum offsets of the two children
    dneg = (kneg-kboxnow)*dk
    if len(WPxkpLists[xboxnow,kneg]) > 0:
        #the new negative child cancels an existing positive particle in its box; the new
        #positive child is represented by moving ("kinking") that particle to the positive child's box
        kinkparticleindex = WPxkpLists[xboxnow,kneg].pop(-1)
        WPxkp[kinkparticleindex,kIndex] += (kpos-kneg)*dk
        WPxkp[kinkparticleindex,kboxIndex] = np.round(WPxkp[kinkparticleindex,kIndex]/dk+NK)
        WPxkpLists[xboxnow,int(WPxkp[kinkparticleindex,kboxIndex])].append(kinkparticleindex)
        kinkcount += 1
    elif len(WPxknLists[xboxnow,kpos]) > 0:
        #the new positive child cancels an existing negative particle; move that one to kneg
        kinkparticleindex = WPxknLists[xboxnow,kpos].pop(-1)
        WPxkn[kinkparticleindex,kIndex] += (kneg-kpos)*dk
        WPxkn[kinkparticleindex,kboxIndex] = np.round(WPxkn[kinkparticleindex,kIndex]/dk+NK)
        WPxknLists[xboxnow,int(WPxkn[kinkparticleindex,kboxIndex])].append(kinkparticleindex)
        kinkcount += 1
    else: #there were no annihilations, so just add the new pair of particles to all structures.
        if ppnum < IPMULT*IPNUM:
            WPxkp[ppnum,:] = [xnow, know+dpos, xboxnow, kpos, 1]
            WPxkpLists[xboxnow,kpos].append(ppnum)
            if ppnum % skipnum == 0:
                ptracklist.append([[timecounter,TimeSteps]])
            ppnum += 1
        else: ppnumexcess += 1     #FIX (bug J): counted, and reported after the loop
        if npnum < IPMULT*IPNUM:
            WPxkn[npnum,:] = [xnow, know+dneg, xboxnow, kneg, 1]
            WPxknLists[xboxnow,kneg].append(npnum)
            if npnum % skipnum == 0:
                ntracklist.append([[timecounter,TimeSteps]])
            npnum += 1
        else: npnumexcess += 1


#Following for loop begins at timecounter=1 and not zero because we captured t=0 case above during initialization
for timecounter in range(1,TimeSteps):
    ## Spawn Events take place here
    ppnumlast = ppnum #Capture numbers of current particles (adjust the real counters while spawing)
    npnumlast = npnum
    prandval = np.random.uniform(0,1,ppnum)
    nrandval = np.random.uniform(0,1,npnum)

    #POSITIVE parents
    for pindex in range(ppnumlast):
        if WPxkp[pindex,aliveIndex] != 1: continue                     #FIX (bug F): dead particles do not spawn
        xnow, know, xboxnow, kboxnow = WPxkp[pindex,(xIndex, kIndex, xboxIndex,kboxIndex)]
        cosval = np.cos(2*nharm*np.pi*xnow/LX)
        if prandval[pindex] < FracSpawn*abs(cosval):                     #FIX (bug A): |cos|, sign used below
            spawn_pair(+1, xnow, know, int(round(xboxnow)), int(round(kboxnow)), np.sign(cosval), timecounter)

    #NEGATIVE parents
    for pindex in range(npnumlast):
        if WPxkn[pindex,aliveIndex] != 1: continue                     #FIX (bug F)
        xnow, know, xboxnow, kboxnow = WPxkn[pindex,(xIndex, kIndex, xboxIndex,kboxIndex)]   #FIX (bug B): was WPxkp
        cosval = np.cos(2*nharm*np.pi*xnow/LX)
        if nrandval[pindex] < FracSpawn*abs(cosval):                     #FIX (bug A)
            spawn_pair(-1, xnow, know, int(round(xboxnow)), int(round(kboxnow)), np.sign(cosval), timecounter)

    ## Propagation Step - All particles move along their phase space Newtonian paths
    xlastp = WPxkp[0:ppnum,xIndex].copy()          #FIX (bug D): copies, not numpy views
    klastp = WPxkp[0:ppnum,kIndex].copy()
    xboxlastp = WPxkp[0:ppnum,xboxIndex].copy()
    kboxlastp = WPxkp[0:ppnum,kboxIndex].copy()
    WPxkp[0:ppnum,xIndex] = klastp*hbar*np.sin(sqrtvmq*dt/m )/sqrtvmq + xlastp*np.cos(sqrtvmq*dt/m)
    WPxkp[0:ppnum,kIndex] = klastp*np.cos(sqrtvmq*dt/m) - xlastp*sqrtvmq*np.sin(sqrtvmq*dt/m)/hbar
    WPxkp[0:ppnum,(xboxIndex,kboxIndex)]=np.round(WPxkp[0:ppnum,(xIndex,kIndex)]/divider_vec+offset_vec)

    xlastn = WPxkn[0:npnum,xIndex].copy()          #FIX (bug D)
    klastn = WPxkn[0:npnum,kIndex].copy()
    xboxlastn = WPxkn[0:npnum,xboxIndex].copy()
    kboxlastn = WPxkn[0:npnum,kboxIndex].copy()
    WPxkn[0:npnum,xIndex] = klastn*hbar*np.sin(sqrtvmq*dt/m )/sqrtvmq + xlastn*np.cos(sqrtvmq*dt/m)
    WPxkn[0:npnum,kIndex] = klastn*np.cos(sqrtvmq*dt/m) - xlastn*sqrtvmq*np.sin(sqrtvmq*dt/m)/hbar
    WPxkn[0:npnum,(xboxIndex,kboxIndex)]=np.round(WPxkn[0:npnum,(xIndex,kIndex)]/divider_vec+offset_vec)

    pchangelist=np.where(((xboxlastp[:] != WPxkp[0:ppnum,xboxIndex]) | (kboxlastp[:] != WPxkp[0:ppnum,kboxIndex])) & (WPxkp[0:ppnum,aliveIndex] == 1) )[0]
    nchangelist=np.where(((xboxlastn[:] != WPxkn[0:npnum,xboxIndex]) | (kboxlastn[:] != WPxkn[0:npnum,kboxIndex])) & (WPxkn[0:npnum,aliveIndex] == 1))[0]

    for pparticleIndex in pchangelist:
        WPxkpLists[int(round(xboxlastp[pparticleIndex])),int(round(kboxlastp[pparticleIndex]))].remove(pparticleIndex)
        xboxnow = int(round(WPxkp[pparticleIndex,xboxIndex]))
        kboxnow = int(round(WPxkp[pparticleIndex,kboxIndex]))
        if not (0 <= xboxnow < NX and 0 <= kboxnow < 2*NK):
            WPxkp[pparticleIndex,aliveIndex] = 0; continue           #FIX: left the represented patch
        if len(WPxknLists[xboxnow,kboxnow]) > 0:
            ppopped = WPxknLists[xboxnow,kboxnow].pop(-1)
            WPxkp[pparticleIndex,aliveIndex] = 0
            WPxkn[ppopped,aliveIndex] = 0
            annihilcount += 1
            for idx, tl in ((pparticleIndex, ptracklist), (ppopped, ntracklist)):   #FIX (bug H)
                slot = track_slot(idx)
                if slot is not None and slot < len(tl): tl[slot][-1][1] = timecounter
        else:
            WPxkpLists[xboxnow,kboxnow].append(pparticleIndex)

    for pparticleIndex in nchangelist:             #FIX (bug E): no longer nested inside the positive loop
        if WPxkn[pparticleIndex,aliveIndex] != 1: continue   #may have been cancelled just above
        WPxknLists[int(round(xboxlastn[pparticleIndex])),int(round(kboxlastn[pparticleIndex]))].remove(pparticleIndex)
        xboxnow = int(round(WPxkn[pparticleIndex,xboxIndex]))
        kboxnow = int(round(WPxkn[pparticleIndex,kboxIndex]))
        if not (0 <= xboxnow < NX and 0 <= kboxnow < 2*NK):
            WPxkn[pparticleIndex,aliveIndex] = 0; continue
        if len(WPxkpLists[xboxnow,kboxnow]) > 0:
            ppopped = WPxkpLists[xboxnow,kboxnow].pop(-1)
            WPxkn[pparticleIndex,aliveIndex] = 0
            WPxkp[ppopped,aliveIndex] = 0
            annihilcount += 1
            for idx, tl in ((pparticleIndex, ntracklist), (ppopped, ptracklist)):
                slot = track_slot(idx)
                if slot is not None and slot < len(tl): tl[slot][-1][1] = timecounter
        else:
            WPxknLists[xboxnow,kboxnow].append(pparticleIndex)

    pWPxTrack[timecounter,0:WPxkp[0:ppnum:skipnum,xIndex].size] = WPxkp[0:ppnum:skipnum,xIndex]
    nWPxTrack[timecounter,0:WPxkn[0:npnum:skipnum,xIndex].size] = WPxkn[0:npnum:skipnum,xIndex]
print("--- %s seconds ---" % (time.time() - start_time))
print("kinks at birth = %d, cancellations on box change = %d, spawns dropped for lack of array space = (%d, %d)"
      % (kinkcount, annihilcount, ppnumexcess, npnumexcess))


In [ ]:
print("Final number of positive and negative particles = (%d, %d)"%(ppnum,npnum))
print("Number of timesteps to be processed, TimeSteps = %d"% TimeSteps)
print("Array space reserved for pos and neg particle tracks = (%s, %s)"% (pWPxTrack.shape, nWPxTrack.shape))
print("Number of pos and neg particles currently being tracked = (%d, %d)"%(len(ptracklist),len(ntracklist)))

Execution times for No Spawn versions:

It took 0.335 seconds to compute 1000 time steps when using 1e5 particles for the zero potential case

It took 5.676 seconds to compute 1000 time steps when using 1e5 particles for the linear potential case

It took 12.589 seconds to compute 1000 time steps when using 1e5 particles for the quadratic potential case

Execution time for Sin Spawn verions at various pre-completion development points.

It took 54.4995629787 secondsto compute 1000 time steps when using 1e5 particles for the quadratic potential case with new spawn enhanced tracking structure initialization added.

In the case of a non-spawning potential field, the following plot shows the trajectories of a sample of about 100 of the particles selected from out of the 1E5 particles used in this simulation. Since we have a non-zero field problem, all the trajectories are criss-crossing curved lines based upon the position and momentum assigned by the Wigner phase space distribution function of the initial wavefunction. 

For spawing fields (greater than quadratic order) our tracks will become interupted by pair annihilations, and new tracks will start due to pair creation.

I will preserve below for reference in a script magic deactivated section of code the original plot code that handled the no-spawing case. It will be followed by the code needed to handle all the complexity that is introduced by appearance and disappearance of tracks. 

In [ ]:
%%script false --no-raise-error
fig10, ax10 = pl.subplots(figsize=(15, 10))
datacols= np.shape(WPxTrack)[1]
for linecount in range(0,datacols):
    ax10.plot(np.arange(TimeSteps)*dt,WPxTrack[0:TimeSteps,linecount])
ax10.set_title('Selected Tracks from the Particle Trajectory Simulation',fontsize= 30)
ax10.legend(loc='upper left')
ax10.set_xlabel('Time (s)')
ax10.set_ylabel('x (m)')
#ax10.set_xlim(xmin=yrs[0], xmax=yrs[-1])
fig10.tight_layout()

In [ ]:
fig10, ax10 = pl.subplots(figsize=(15, 10))
#loop through all start/stop lists in ptracklist
slotcount = -1
for startstops in ptracklist:
    slotcount+=1
    #print("slotcount = %d, startstoplist = %s"%(slotcount,startstops))
    for segment in startstops:
        #print("segment = %s"% segment)
        ax10.plot(np.arange(segment[0],segment[1])*dt,pWPxTrack[segment[0]:segment[1],slotcount],"-b")
#Similarly for the negative particle tracks
slotcount = -1
for startstops in ntracklist:
    slotcount+=1
    #print("slotcount = %d, startstoplist = %s"%(slotcount,startstops))
    for segment in startstops:
        #print("segment = %s"% segment)
        ax10.plot(np.arange(segment[0],segment[1])*dt,nWPxTrack[segment[0]:segment[1],slotcount],"-r")
        
ax10.set_title('Selected Tracks from the Particle Trajectory Simulation',fontsize= 30)
ax10.legend(loc='upper left')
ax10.set_xlabel('Time (s)')
ax10.set_ylabel('x (m)')
#ax10.set_xlim(xmin=yrs[0], xmax=yrs[-1])
fig10.tight_layout()

In the above simulation enough time passed that some of our particles may have propagated outside the x extremes of our represented patch of phase space. In problems with a non-zero field, we can also have particles cross outside the k boundaries as well. So, here is a little bookkeepping code that will come in handy in future versions of this piece of code to assess the numbers of those exiting particles.

First we generate the post simulation signed mass histogram.

In [ ]:
WPxkPostSignedMassHist = MakeWignerHist(WPxkp,WPxkn) 

Now we compare the total number of signed particles (mass equivalent) of before and after simulation.

In [ ]:
WPxkSignedMassHistOrigTotal = WPxkSignedMassHistOrig.sum()
WPxkSignedMassHistPostTotal = WPxkPostSignedMassHist.sum()
print("WPSignedMassHistOrigTotal = ", WPxkSignedMassHistOrigTotal)
print("WPSignedMassHistPostTotal = ", WPxkSignedMassHistPostTotal)
print("Particles that exited the simulation space = %d" %(WPxkSignedMassHistOrigTotal-WPxkSignedMassHistPostTotal))

We see that in the present case, all trajectories were confined to our xk simulation space patch.

## Examine the new x and k distributions and Compare with Analytic Solutions. 

In [ ]:
pxdat = WPxkp[0:ppnum,xIndex]/dx
pydat = WPxkp[0:ppnum,kIndex]/dk
pdata = np.column_stack((pxdat, pydat))

nxdat = WPxkn[0:npnum,xIndex]/dx
nydat = WPxkn[0:npnum,kIndex]/dk
ndata = np.column_stack((nxdat,nydat))

fig, (ax1, ax2, ax3) = pl.subplots(nrows=1, ncols=3,figsize=(12, 4))

ax1.scatter(x=pxdat, y=pydat, marker='o', c='b', edgecolor='b')
ax1.scatter(x=nxdat, y=nydat, marker='o', c='r', edgecolor='r')
ax1.set_title('$k$ vs $x$ in units of $dk$ and $dx$')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$k$')

_, bins, _ = ax2.hist(pxdat, bins=50, range=[pdata.min(), pdata.max()], density=True)
ax2.hist(pydat, bins=bins, alpha=0.5, density=True)
ax2.legend(loc=(0.65, 0.8))
ax2.set_title('Frequency of $x$, $k$ in units of $dk$, $dx$')
ax2.yaxis.tick_right()

_, bins, _ = ax3.hist(nxdat, bins=50, range=[ndata.min(), ndata.max()], density=True)
ax3.hist(nydat, bins=bins, alpha=0.5, density=True)
ax3.legend(loc=(0.65, 0.8))
ax3.set_title('Frequency of $x$, $k$ in units of $dk$, $dx$')
ax3.yaxis.tick_right()

In [ ]:
surfWPxkPostEvolution = go.Surface(x=x,y=k,z=np.transpose(WPxkPostSignedMassHist), colorbar = dict(exponentformat = 'power')) 
dataWPxkPostEvolution = [surfWPxkPostEvolution] 
layoutWPxkPostEvolution = go.Layout(
                    scene = dict(
                    xaxis = dict(
                        title='x (m)', exponentformat = 'power'),
                    yaxis = dict(
                        title='k (momentum)', exponentformat = 'power'),
                    zaxis = dict(
                        title='Post Evolution Wigner Distribution in Signed Mass Counts', exponentformat = 'power'),),
                    width=700,
                    margin=dict(
                    r=20, b=10,
                    l=10, t=10)                    
)
    
figWPxkPostEvolution = go.Figure(data=dataWPxkPostEvolution, layout=layoutWPxkPostEvolution)
iplot(figWPxkPostEvolution)

Generate and plot the x marginal densities before and after the evolution

In [ ]:
WPxSignedMassHistOrig = WPxkSignedMassHistOrig.sum(axis=1)
WPxSignedMassHistPost = WPxkPostSignedMassHist.sum(axis=1)

In [ ]:

WPxCountsOrig_Trace = go.Scatter(
    x = x,
    y = WPxSignedMassHistOrig,
    mode = 'lines',
    name = 'Original'
)
WPxCountsPE_Trace = go.Scatter(
    x = x,
    y = WPxSignedMassHistPost,
    mode = 'lines',
    name = 'PostEvolution'
)

WPxPrePostData = [WPxCountsOrig_Trace, WPxCountsPE_Trace]
#iplot(data)

WPxPrePostLayout = dict(title = 'x Marginal Signed Mass Distribution Pre vs Post Evolution ',
              yaxis = dict(zeroline = False),
              xaxis = dict(zeroline = False),   
              autosize=False,
              width=700,
              height=500,
             )

WPxPrePostFig = dict(data=WPxPrePostData, layout=WPxPrePostLayout)
iplot(WPxPrePostFig)

Compare this to the analytic solutions to the same problem in the graphic below. We have normalized the particle counts and the analytic density so that each value represents the probability of a particle in an interval of size dx with dx being the position discretization value used throughout this worksheet.

The analytic solution for the QHO with an initially displaced Gaussian wavepacket can be found in a particularly simple form in the paper: 

The evolution of oscillator wave functions
by Mark Andrew
arXiv:1509.05968v1

In [ ]:
omega = sqrtvmq/m
#This implements the Quantum Harmonic Oscillator Schrodinger eq solution for an initially 
#displaced Gaussian wavepack, but I have stripped off the complicated phase factor as it
#was going to get stripped off anyway in the next workbook cell when we form the Probability density
def QHOGaussianEvolution(t, x, x0, a, k0, hbar, m):
    return ((2*np.pi)**(-1.0/4.0)/np.sqrt(a))*np.exp(-(x-x0*np.cos(omega*t))**2/(4.0*(a**2)))

In [ ]:
AnalyticOrig=np.power(abs(QHOGaussianEvolution(0, x, x0, a, k0, hbar, m)),2.0)
AnalyticPost=np.power(abs(QHOGaussianEvolution(TimeSteps*dt, x, x0, a, k0, hbar, m)),2.0)

In [ ]:
AnalyticOrig_Trace = go.Scatter(
    x = x,
    y = AnalyticOrig*dx,
    mode = 'lines',
    name = 'Analytic Original',
    line = dict(shape = 'linear', dash = 'dash', width= 6,)
)
AnalyticPost_Trace = go.Scatter(
    x = x,
    y = AnalyticPost*dx,
    #mode = 'lines',
    name = 'Analytic Post',
    line = dict(shape = 'linear', dash = 'dash', width= 6,)
)
WPxCountsOrig_Trace = go.Scatter(
    x = x,
    y = WPxSignedMassHistOrig/pnum,
    mode = 'lines',
    name = 'Particles Original'
)
WPxCountsPE_Trace = go.Scatter(
    x = x,
    y = WPxSignedMassHistPost/pnum,
    mode = 'lines',
    name = 'Particles Post'
)

AnalyticPrePostData = [AnalyticOrig_Trace, AnalyticPost_Trace, WPxCountsOrig_Trace, WPxCountsPE_Trace]

AnalyticPrePostLayout = dict(title = 'x Marginal Signed Mass Distribution Pre vs Post Evolution ',
              yaxis = dict(zeroline = False),
              xaxis = dict(zeroline = False)
             )

AnalyticPrePostFig = dict(data=AnalyticPrePostData, layout=AnalyticPrePostLayout)
iplot(AnalyticPrePostFig)

We can also compare the Momentum statistics of our particle collection with that of the analytic solution for the Gaussian packet in a quadratic potential (the solution for which is found in paper cited above.

First form the momentum histograms of the particles before and after evolution

In [ ]:
WPkSignedMassHistOrig = WPxkSignedMassHistOrig.sum(axis=0)
WPkSignedMassHistPost = WPxkPostSignedMassHist.sum(axis=0)

Let's just check the normalization of histogram to make sure we've done it right to this point.

In [ ]:
WPkSignedMassHistOrig.sum()/WPxkSignedMassHistOrig.sum()  #FIX (bug L): pnum was undefined

The analytic solution for the momentum distribution for an acclerating wavepacket solution is given by the following expression.  The analytic solution can be derived from information in the paper cited above (from the mean momentum oscillation and the fact that the displaced Gaussian wavefunction density is invariant in the special case of a QHO).  

In [ ]:
def kQHOProbAccGaussianEvolution(t, k, a, k0, hbar, m):
    return a*np.sqrt(2.0)/(np.sqrt(np.pi))*np.exp(-2.0*(a*((k+m*omega/hbar*x0*np.sin(omega*t))))**2)

In [ ]:
kAnalyticOrig=kQHOProbAccGaussianEvolution(0, k, a, k0, hbar, m)
kAnalyticPost=kQHOProbAccGaussianEvolution(TimeSteps*dt, k, a, k0, hbar, m)

In [ ]:
AnalyticOrig_Trace = go.Scatter(
    x = x,
    y = kAnalyticOrig*dk,
    mode = 'lines',
    name = 'Analytic Original',
    line = dict(shape = 'linear', dash = 'dash', width= 6,)
)
AnalyticPost_Trace = go.Scatter(
    x = x,
    y = kAnalyticPost*dk,
    #mode = 'lines',
    name = 'Analytic Post',
    line = dict(shape = 'linear', dash = 'dash', width= 6,)
)
WPkCountsOrig_Trace = go.Scatter(
    x = x,
    y = WPkSignedMassHistOrig/pnum,
    mode = 'lines',
    name = 'Particles Original'
)
WPkCountsPE_Trace = go.Scatter(
    x = x,
    y = WPkSignedMassHistPost/pnum,
    mode = 'lines',
    name = 'Particles Post'
)

AnalyticPrePostData = [AnalyticOrig_Trace, AnalyticPost_Trace, WPkCountsOrig_Trace, WPkCountsPE_Trace]

AnalyticPrePostLayout = dict(title = 'k Marginal Signed Mass Momentum Distribution Pre vs Post Evolution vs Analytic Solutions',
              yaxis = dict(zeroline = False),
              xaxis = dict(zeroline = False)
             )

AnalyticPrePostFig = dict(data=AnalyticPrePostData, layout=AnalyticPrePostLayout)
iplot(AnalyticPrePostFig)

Once again we see essentially a perfect match just as we expect for the Gaussian Wavepacket. But also once again, from the mathematics we know that this is just one example of the more general statement: The purely Newtonian, curved line, propagation of real particles (as distated by the Wigner/Liouville evolution equation) will reproduce both the position and the momentum distributions obtained from a Schrodinger equation solution for **any initial wavefunction** in the quadratic potential field case.

For potentials of higher than second degree the Wigner/Liouville evolution equation introduces a **collision** term. This parallels the introduction of a collision term in the Boltzman equations when particles can collide in a gas and thus exchange momentum and so introduces diffusion into the evolution.  

It has been shown in previous workbooks how the collision term of the Wigner/Liouville evolution equation behaves as if the particles (which are entire worlds) behave as if they collide with the surface of a background sea, scattering some of the sea's otherwise ground state particles into positive and negative mass pairs. This can be likened to scattering a droplet off the surface of a fluid, causing postive excusions and negative troughs in the fluid. No wonder Couder et al. found some resemblance to QM behavior in their bouncing droplet experiments!!

We will now begin implementing v4 of this series that will consider third order potentials in order to illustrate the simplest case in which these collision events and the associated particle pair excitations arise.

**Added for the review: comparison with the exact solution of the same problem.** The spawn amplitude corresponds to a sinusoidal potential energy $V_p\sin(2\,n_{harm}\pi x/L_X)$ with $V_p = \mathrm{FracSpawn}\,\hbar/dt$. The reference is a split-operator Schrödinger solution for $U(x) = q_e V_2 x^2 + V_p\sin(2 n_{harm}\pi x/L_X)$; its $|\psi|^2$ is the exact position marginal.

In [ ]:
#Added for the review: exact reference for the potential actually simulated
Vp = FracSpawn*hbar/dt
qmode = 2*nharm*np.pi/LX
NS = 4*NX; xs = (np.arange(NS)-NS//2)*(LX/NS); dxs = LX/NS
kks = 2*np.pi*np.fft.fftfreq(NS, d=dxs)
Us = q_e*V2*xs**2 + Vp*np.sin(qmode*xs)
psi = (2*np.pi*a**2)**-0.25*np.exp(-(xs-x0)**2/(4*a**2))
half = np.exp(-0.5j*Us*dt/hbar); kin = np.exp(-0.5j*hbar*kks**2*dt/m)
for i in range(1, TimeSteps):
    psi = half*np.fft.ifft(kin*np.fft.fft(half*psi))
rho = np.abs(psi)**2
#bin onto the particle boxes x[i] +- dx/2
cum = np.concatenate([[0.0], np.cumsum(rho)*dxs]); xe = np.concatenate([xs-dxs/2, [xs[-1]+dxs/2]])
edges = np.concatenate([x-dx/2, [x[-1]+dx/2]])
rho_ref = np.diff(np.interp(edges, xe, cum))          #probability per box
rho_par = WPxkPostSignedMassHist.sum(axis=1)/WPxkSignedMassHistOrig.sum()
print("Vp = %.3e J = %.3f meV" % (Vp, Vp/q_e*1e3))
print("L1 distance, particles vs exact = %.3f   (%d net particles; see the review for the noise level)"
      % (np.abs(rho_par-rho_ref).sum(), WPxkSignedMassHistOrig.sum()))
print("mean x: particles %.4e   exact %.4e" % ((x*rho_par).sum(), (x*rho_ref).sum()))
fig, axr = pl.subplots(figsize=(8,4))
axr.plot(x, rho_ref, 'k', label='exact $|\\psi|^2$')
axr.step(x, rho_par, where='mid', label='signed particles')
axr.set_xlim(x0-10*a, -x0+10*a); axr.set_xlabel('x (m)'); axr.legend()
